# Практическая работа 3. Функции, данные и шум

Версия методички: **v2**.

Работа объединяет две соседние темы курса: **функцию и её дискретное представление** и **данные и шум**. В работе 1 формулы уже вычислялись для массивов и строились графики; теперь основная цель — научиться оформлять повторяющиеся вычисления как собственные функции, а затем использовать эти функции для работы с конечными наборами данных.

Вторая линия работы — количественное описание разброса и моделирование шума. Сначала стандартное отклонение разбирается на обычном детерминированном массиве, и только после этого используется как параметр нормального распределения.

Основная цепочка:

$$\text{формула}\rightarrow\text{функция Python}\rightarrow\text{дискретное представление}\rightarrow\text{характеристики разброса}\rightarrow\text{случайная выборка}\rightarrow\text{зашумлённые данные}\rightarrow\text{статистическое описание}.$$

Из предыдущих работ считаются известными NumPy-массивы, индексация, `np.arange`, векторные выражения, `np.sin`/`np.cos`, построение графиков, `sum`, `mean`, `min/max`, анализ и отладка кода.

# План работы

1. Собственные функции Python.
2. Дискретное представление функции и равномерная сетка.
3. Статистики как вычисления над массивом.
4. Случайные данные, нормальное распределение и шум.
5. Квантили и гистограмма распределения.
6. Повторно используемое статистическое описание и эксперимент с уровнем шума.
7. Самостоятельная работа.
8. Контрольные задания.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Собственные функции Python

Собственная функция отделяет правило вычисления от конкретного места, где это правило используется. В этой работе функции нужны не ради нового синтаксиса как такового: дальше в функции будут вынесены математическая модель, вычисление статистик и операции с шумом.

### Пример 1. Функция одного аргумента

**Задача.** Оформить зависимость $y=2x+1$ как собственную функцию Python и вычислить её в нескольких точках.

**Уже используемые concepts:** —.

**Новые concepts:** `python_function`, `function_parameter`, `return_value`.

#### Объяснение

- `python_function` — функция Python: именованный блок кода, который можно вызывать многократно.
- `function_parameter` — параметр функции: имя входного значения внутри определения функции.
- `return_value` — возвращаемое значение: результат, передаваемый из функции вызывающему коду командой `return`.

При вызове `linear(3)` число 3 становится конкретным входным значением параметра `x`.

#### Математическая модель

$$f(x)=2x+1.$$

#### Числовой пример

$$f(0)=1,\qquad f(3)=7.$$

#### Код решения

In [ ]:
def linear(x):
    return 2 * x + 1

print("linear(0) =", linear(0))
print("linear(3) =", linear(3))

#### Разбор кода

После `def` задаются имя функции и параметр `x`. Тело функции вычисляет выражение, а `return` возвращает результат. Функция не обязана сама печатать или строить график: внешний код решает, что делать с возвращённым значением.

### Пример 2. Параметризованная функция

**Задача.** Написать одну функцию для семейства квадратичных зависимостей

$$P(x)=ax^2+bx+c.$$

**Уже используемые concepts:** `python_function`, `function_parameter`, `return_value`, `polynomial_function`.

**Новые concepts:** `multiple_parameters`, `function_reuse`.

#### Объяснение

- `multiple_parameters` — несколько параметров функции позволяют передавать все коэффициенты отдельно.
- `function_reuse` — одно определение функции используется для разных наборов входных данных.

#### Математическая модель

$$P(x;a,b,c)=ax^2+bx+c.$$

#### Числовой пример

При $x=2$, $a=1$, $b=-3$, $c=2$:

$$P(2)=4-6+2=0.$$

#### Код решения

In [ ]:
def polynomial(x, a, b, c):
    return a * x**2 + b * x + c

print(polynomial(2, 1, -3, 2))
print(polynomial(2, 2, 0, 1))

#### Разбор кода

Коэффициенты не зашиты в тело функции. Поэтому один алгоритм подходит для всех вариантов, а различаются только значения параметров.

**Итог раздела.** Функция хранит правило вычисления, а параметры отделяют общий алгоритм от конкретных чисел.

# 2. Дискретное представление функции и равномерная сетка

Математическая функция задаёт правило для аргумента, а программа обычно работает с конечным набором значений. Табулирование создаёт пары $(x_i,f(x_i))$ — дискретное представление функции на выбранной сетке.

### Пример 3. Дискретное представление функции

**Задача.** Передать NumPy-массив в собственную функцию и получить массив результатов без явного цикла.

**Уже используемые concepts:** `python_function`, `numpy_array`, `vectorized_expression`, `plot`, `numpy_arange`, `grid`, `function_parameter`, `return_value`, `multiple_parameters`, `polynomial_function`.

**Новые concepts:** `array_function_argument`, `function_tabulation`.

#### Объяснение

- `array_function_argument` — NumPy-массив можно передать как вход собственной функции.
- `function_tabulation` — табулирование: вычисление функции на конечном наборе аргументов; полученные пары являются её дискретным представлением.

#### Математическая модель

$$y_i=P(x_i),\qquad i=0,1,\ldots,N-1.$$

#### Числовой пример

Для $x=[-1,0,1]$ и $P(x)=x^2-2x-3$:

$$y=[0,-3,-4].$$

#### Код решения

In [ ]:
x = np.arange(-2, 3, 1)
y = polynomial(x, 1, -2, -3)

print("x =", x)
print("y =", y)

plt.plot(x, y, marker="o")
plt.xlabel("x")
plt.ylabel("P(x)")
plt.grid(True)
plt.show()

#### Разбор кода

Внутри `polynomial` используются только векторные операции NumPy, поэтому формула автоматически применяется ко всем элементам массива `x`. Отдельный цикл для табулирования не нужен.

### Пример 4. Равномерная сетка `np.linspace`

**Задача.** Получить 9 равномерно расположенных точек на отрезке $[-2,2]$, вычислить шаг и значения функции $x^2$.

**Уже используемые concepts:** `numpy_array`, `function_tabulation`, `indexing`, `python_function`, `function_parameter`, `return_value`, `array_function_argument`, `vectorized_expression`.

**Новые concepts:** `numpy_linspace`, `uniform_grid`, `grid_step`.

#### Объяснение

- `numpy_linspace` — `np.linspace(a,b,N)` создаёт $N$ равномерно расположенных значений и по умолчанию включает обе границы.
- `uniform_grid` — равномерная сетка: соседние значения аргумента разделены одинаковым расстоянием.
- `grid_step` — шаг сетки, то есть расстояние между соседними значениями аргумента.

#### Математическая модель

Для включённых границ:

$$h=\frac{b-a}{N-1}.$$

#### Числовой пример

Для $a=-2$, $b=2$, $N=9$:

$$h=\frac{4}{8}=0.5.$$

#### Код решения

In [ ]:
def square(x):
    return x**2

x = np.linspace(-2, 2, 9)
y = square(x)

print("x =", x)
print("h =", x[1] - x[0])
print("y =", y)

#### Разбор кода

В `linspace` третье число — количество точек, а не шаг. Фактический шаг проверяется разностью соседних элементов.

### Пример 5. `np.arange` и `np.linspace`

**Задача.** Сравнить сетку, заданную шагом, и сетку, заданную числом точек.

**Уже используемые concepts:** `numpy_arange`, `numpy_linspace`, `uniform_grid`, `grid_step`.

**Новые concepts:** —.

#### Объяснение

Новых понятий нет: сравниваются два уже изученных способа построения равномерной сетки. Выбор определяется тем, что задано в условии — шаг или число точек.

#### Математическое описание

`np.arange` естественен, когда условие задаёт шаг $h$. `np.linspace` естественен, когда задано число точек $N$ на конечном интервале.

#### Числовой пример

`np.arange(0,1.01,0.25)` и `np.linspace(0,1,5)` в этом примере дают одинаковые числа, но отвечают на разные постановки задачи.

#### Код решения

In [ ]:
x_step = np.arange(0, 1.01, 0.25)
x_count = np.linspace(0, 1, 5)
print("arange  =", x_step)
print("linspace=", x_count)

#### Разбор кода

Выбор функции определяется не внешним видом результата, а тем, какой параметр задан в условии — шаг или число точек.

**Итог раздела.** Дискретное представление функции определяется и формулой, и выбранной сеткой аргумента.

# 3. Статистики как вычисления над массивом

Среднее уже использовалось в первой работе. Здесь оно становится хорошим простым примером собственной функции, после чего тем же способом разбираются характеристики разброса.

### Пример 6. Среднее как собственная функция

**Задача.** Реализовать среднее арифметическое собственной функцией через сумму и длину массива и сравнить результат с `np.mean`.

**Уже используемые concepts:** `python_function`, `array_function_argument`, `sum`, `len`, `mean`, `numpy_array`, `function_parameter`, `return_value`.

**Новые concepts:** —.

#### Объяснение

Среднее уже знакомо из первой работы. Здесь оно используется как простой пример перевода математического определения в собственную функцию над массивом.

#### Математическая модель

$$\mu=\frac{1}{N}\sum_{i=0}^{N-1}x_i.$$

#### Числовой пример

Для $[2,4,6]$ среднее равно $4$.

#### Код решения

In [ ]:
def mean_manual(data):
    return np.sum(data) / len(data)

data = np.array([2.0, 4.0, 5.0, 9.0])
print("manual =", mean_manual(data))
print("numpy  =", np.mean(data))

#### Разбор кода

Функция получает массив, вычисляет сумму всех элементов и делит её на их количество. Здесь цель — не заменить `np.mean`, а потренироваться переводить математическое определение в собственную функцию.

### Пример 7. Дисперсия и стандартное отклонение по определению

**Задача.** Вычислить дисперсию и стандартное отклонение конечного массива сначала по формулам, затем проверить функциями NumPy.

**Уже используемые concepts:** `mean`, `vectorized_expression`, `numpy_array`.

**Новые concepts:** `variance`, `standard_deviation`.

#### Объяснение

- `variance` — дисперсия: среднее квадратов отклонений элементов от среднего.
- `standard_deviation` — стандартное отклонение: квадратный корень из дисперсии; оно имеет те же единицы, что исходные данные.

В этой работе используется нормировка на $N$, как в `np.var` и `np.std` по умолчанию.

#### Математическая модель

$$\sigma^2=\frac{1}{N}\sum_{i=0}^{N-1}(x_i-\mu)^2,$$

$$\sigma=\sqrt{\sigma^2}.$$

#### Числовой пример

Для $[1,2,3]$ имеем $\mu=2$, $\sigma^2=2/3$, $\sigma\approx0.816$.

#### Код решения

In [ ]:
data = np.array([1.0, 2.0, 3.0])
mu = np.mean(data)
var_manual = np.mean((data - mu)**2)
std_manual = var_manual**0.5

print("mean =", mu)
print("var manual / numpy =", var_manual, np.var(data))
print("std manual / numpy =", std_manual, np.std(data))

#### Разбор кода

Выражение `data - mu` сразу формирует все отклонения, квадрат применяется ко всему массиву, а среднее этих квадратов даёт дисперсию. Затем из дисперсии получается стандартное отклонение.

### Пример 8. Дисперсия как собственная функция

**Задача.** Оформить вычисление дисперсии как повторно используемую функцию.

**Уже используемые concepts:** `python_function`, `array_function_argument`, `mean`, `variance`, `standard_deviation`, `numpy_array`, `function_parameter`, `return_value`, `vectorized_expression`.

**Новые concepts:** —.

#### Объяснение

Алгоритм дисперсии уже разобран по шагам. Теперь он оформляется как функция, чтобы применять его к разным массивам без копирования кода.

#### Математическая модель

$$\operatorname{var}(x)=\frac{1}{N}\sum_i(x_i-\mu)^2.$$

#### Числовой пример

Для любого нового массива меняются только данные, а алгоритм остаётся тем же.

#### Код решения

In [ ]:
def variance_manual(data):
    mu = np.mean(data)
    return np.mean((data - mu)**2)

data = np.array([1.0, 2.0, 3.0, 4.0])
var_value = variance_manual(data)
std_value = var_value**0.5
print("var =", var_value)
print("std =", std_value)

#### Разбор кода

Функция скрывает повторяющийся алгоритм, но математическая модель остаётся видимой. В самостоятельной работе аналогично потребуется самостоятельно оформить функцию стандартного отклонения.

**Итог раздела.** Статистическая характеристика — это вычисление над массивом, которое можно сначала понять по формуле, а затем оформить собственной функцией.

# 4. Случайные данные, нормальное распределение и шум

Теперь характеристика `standard_deviation` уже известна. Поэтому её можно осмысленно использовать как параметр распределения при моделировании случайных данных.

### Пример 9. Генератор и воспроизводимая случайная выборка

**Задача.** Создать два генератора с одинаковым начальным состоянием и убедиться, что они дают одинаковые выборки при одинаковом вызове.

**Уже используемые concepts:** `numpy_array`.

**Новые concepts:** `random_generator`, `random_seed`, `random_sample`.

#### Объяснение

- `random_generator` — генератор псевдослучайных чисел NumPy.
- `random_seed` — начальное состояние генератора; одинаковый seed позволяет повторить эксперимент.
- `random_sample` — конечный набор случайно сгенерированных значений.

#### Математическое описание

Конкретные значения случайной выборки заранее не вычисляются по простой детерминированной формуле, но одинаковое начальное состояние воспроизводит ту же псевдослучайную последовательность.

#### Числовой пример

Два генератора с `seed=42`, вызванные одинаково, должны вернуть одинаковые первые шесть значений.

#### Код решения

In [ ]:
rng1 = np.random.default_rng(42)
rng2 = np.random.default_rng(42)

a = rng1.uniform(-1.0, 1.0, size=6)
b = rng2.uniform(-1.0, 1.0, size=6)

print(a)
print(b)

#### Разбор кода

Seed не делает данные «неслучайными» по структуре эксперимента; он делает конкретную псевдослучайную последовательность повторяемой, что удобно для обучения и проверки.

### Пример 10. Воспроизводимая нормальная случайная выборка

**Задача.** Получить большой набор значений из нормального распределения с заданными средним и стандартным отклонением.

**Уже используемые concepts:** `random_generator`, `random_seed`, `random_sample`, `mean`, `standard_deviation`.

**Новые concepts:** `normal_distribution`.

#### Объяснение

`normal_distribution` — нормальное распределение, задаваемое средним $\mu$ и стандартным отклонением $\sigma$.

#### Математическая модель

$$\varepsilon_i\sim\mathcal{N}(\mu,\sigma^2).$$

В вызове `rng.normal(loc=mu, scale=sigma, size=M)` параметр `scale` — именно стандартное отклонение $\sigma$.

#### Числовой пример

Для большой выборки из $\mathcal{N}(0,0.5^2)$ выборочное среднее обычно получается близким к 0, а `np.std` — близким к 0.5, но не обязано совпадать с ними точно.

#### Код решения

In [ ]:
rng = np.random.default_rng(123)
data = rng.normal(loc=0.0, scale=0.5, size=1000)
print("mean =", np.mean(data))
print("std =", np.std(data))

#### Разбор кода

`loc` задаёт центр, `scale` — стандартное отклонение распределения, `size` — число наблюдений. Статистики конечной выборки колеблются около параметров распределения.

### Пример 11. Зашумлённые данные и остаток

**Задача.** Добавить нормальный шум к чистым отсчётам функции и вычислить остаток между наблюдаемыми и чистыми данными.

**Уже используемые concepts:** `random_generator`, `normal_distribution`, `numpy_array`, `numpy_linspace`, `mean`, `standard_deviation`, `random_sample`, `random_seed`, `python_function`, `function_parameter`, `return_value`, `array_function_argument`, `shape`, `vectorized_expression`.

**Новые concepts:** `additive_noise`, `noisy_data`, `residual`.

#### Объяснение

- `additive_noise` — аддитивный шум: случайная величина прибавляется к исходному значению.
- `noisy_data` — зашумлённые данные после добавления шума.
- `residual` — остаток: разность между наблюдаемым и опорным значением.

#### Математическая модель

$$y_i^{(noisy)}=y_i^{(clean)}+\varepsilon_i,$$

$$r_i=y_i^{(noisy)}-y_i^{(clean)}.$$

В этой модели $r_i=\varepsilon_i$.

#### Числовой пример

Если чистое значение равно 5.0, а шум равен $-0.3$, наблюдаемое значение равно 4.7, а остаток — $-0.3$.

#### Код решения

In [ ]:
def add_normal_noise(y, sigma, seed):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, sigma, size=y.shape)
    return y + noise

x = np.linspace(0, 2*np.pi, 80)
y_clean = 2*np.sin(x)
y_noisy = add_normal_noise(y_clean, sigma=0.4, seed=10)
r = y_noisy - y_clean

print("residual mean =", np.mean(r))
print("residual std =", np.std(r))

#### Разбор кода

Функция добавления шума получает уже вычисленный массив `y`, поэтому не зависит от конкретной математической функции. Размер шума берётся из `y.shape`, и каждому чистому отсчёту соответствует одно случайное отклонение.

**Итог раздела.** После введения стандартного отклонения параметр `sigma` генератора имеет ясный статистический смысл, а остаток позволяет анализировать добавленный шум отдельно от исходной зависимости.

# 5. Квантили и гистограмма распределения

Среднее и стандартное отклонение дают компактное числовое описание. Квантили показывают положение частей упорядоченного набора, а гистограмма — как наблюдения распределены по диапазону значений.

### Пример 12. Квантили и межквартильный размах

**Задача.** Вычислить 25%, 50% и 75% квантили и межквартильный размах.

**Уже используемые concepts:** `numpy_array`.

**Новые concepts:** `quantile`, `median`, `interquartile_range`.

#### Объяснение

- `quantile` — значение, ниже которого находится заданная доля данных.
- `median` — квантиль уровня 0.5.
- `interquartile_range` — межквартильный размах $Q_{0.75}-Q_{0.25}$.

#### Математическая модель

$$IQR=Q_{0.75}-Q_{0.25}.$$

#### Числовой пример

Для упорядоченного набора $[1,2,3,4,5]$ медиана равна 3.

#### Код решения

In [ ]:
data = np.array([1, 2, 2, 3, 4, 8, 20], dtype=float)
q25, q50, q75 = np.quantile(data, [0.25, 0.50, 0.75])
iqr = q75 - q25
print("q25 =", q25)
print("median =", q50)
print("q75 =", q75)
print("IQR =", iqr)
print("mean =", np.mean(data))

#### Разбор кода

`np.quantile` получает несколько уровней сразу. Сравнение медианы и среднего показывает, что разные характеристики центра реагируют на крайние значения по-разному.

### Пример 13. Гистограмма распределения

**Задача.** Построить гистограмму случайных данных с заданным числом интервалов и объяснить смысл высоты столбцов.

**Уже используемые concepts:** `random_sample`, `random_generator`, `normal_distribution`, `random_seed`.

**Новые concepts:** `frequency_distribution`, `histogram`, `histogram_bins`.

#### Объяснение

- `frequency_distribution` — описание того, как часто значения встречаются в разных частях диапазона.
- `histogram` — гистограмма: графическое представление частотного распределения по интервалам.
- `histogram_bins` — интервалы гистограммы (`bins`).

Гистограмма отличается от `bar`: `bar` показывает отдельные заранее заданные величины, а гистограмма сначала группирует числовые наблюдения по интервалам.

#### Математическое описание

Диапазон разбивается на интервалы $B_k$, затем считается

$$n_k=\#\{x_i:x_i\in B_k\}.$$

#### Числовой пример

Для данных `[0.1,0.2,0.8,1.2]` и интервалов `[0,0.5)` и `[0.5,1.5]` частоты равны 2 и 2.

#### Код решения

In [ ]:
rng = np.random.default_rng(5)
data = rng.normal(0.0, 1.0, size=1500)

plt.hist(data, bins=20)
plt.xlabel("value")
plt.ylabel("count")
plt.show()

#### Разбор кода

`plt.hist` разбивает диапазон данных на `bins` интервалов и считает число наблюдений в каждом. Сам параметр `bins` будет исследован отдельно в самостоятельной работе.

**Итог раздела.** Квантили дают числовые границы частей набора, а гистограмма показывает частотную структуру распределения.

# 6. Повторно используемое статистическое описание и эксперимент

Последний шаг — собрать уже известные операции в собственную функцию и применить её в небольшом эксперименте. Новая сложность здесь не статистическая, а структурная: одна функция возвращает несколько связанных результатов.

### Пример 14. Функция статистического описания

**Задача.** Написать `describe_data(data)`, возвращающую среднее, стандартное отклонение, минимум, максимум и три квартиля.

**Уже используемые concepts:** `python_function`, `mean`, `standard_deviation`, `min_max`, `quantile`, `numpy_array`, `function_parameter`, `return_value`, `array_function_argument`, `median`.

**Новые concepts:** `multiple_return_values`, `tuple_unpacking`.

#### Объяснение

- `multiple_return_values` — функция возвращает несколько связанных результатов одной командой `return`.
- `tuple_unpacking` — эти результаты можно присвоить нескольким переменным слева.

#### Математическая модель

Функция возвращает

$$\mu,\;\sigma,\;x_{min},\;x_{max},\;Q_{0.25},\;Q_{0.5},\;Q_{0.75}.$$

#### Числовой пример

Для `[1,2,3]` среднее равно 2, минимум 1, максимум 3; остальные характеристики вычисляются тем же вызовом.

#### Код решения

In [ ]:
def describe_data(data):
    q25, q50, q75 = np.quantile(data, [0.25, 0.50, 0.75])
    return np.mean(data), np.std(data), np.min(data), np.max(data), q25, q50, q75

data = np.array([1, 2, 4, 5, 8], dtype=float)
mean_value, std_value, min_value, max_value, q25, q50, q75 = describe_data(data)
print(mean_value, std_value, min_value, max_value, q25, q50, q75)

#### Разбор кода

Внутри функции собраны уже знакомые операции. Распаковка позволяет дать каждому возвращённому числу понятное имя во внешнем коде.

### Пример 15. Сравнение уровней шума

**Задача.** При одинаковом seed сравнить два уровня нормального шума по стандартному отклонению и гистограмме.

**Уже используемые concepts:** `normal_distribution`, `standard_deviation`, `histogram`, `random_seed`, `legend`, `random_generator`, `random_sample`, `histogram_bins`, `multiple_series`, `frequency_distribution`.

**Новые concepts:** `parameter_experiment`.

#### Объяснение

`parameter_experiment` — параметрический эксперимент: целенаправленно изменяется один параметр, а остальные условия сохраняются.

#### Математическая модель

$$\varepsilon^{(1)}\sim\mathcal{N}(0,0.3^2),\qquad
\varepsilon^{(2)}\sim\mathcal{N}(0,0.9^2).$$

#### Числовой пример

Второй набор должен иметь примерно втрое большее стандартное отклонение и более широкую гистограмму.

#### Код решения

In [ ]:
rng1 = np.random.default_rng(7)
rng2 = np.random.default_rng(7)
noise_small = rng1.normal(0.0, 0.3, size=1000)
noise_large = rng2.normal(0.0, 0.9, size=1000)

print("std small =", np.std(noise_small))
print("std large =", np.std(noise_large))

plt.hist(noise_small, bins=20, alpha=0.6, label="sigma = 0.3")
plt.hist(noise_large, bins=20, alpha=0.6, label="sigma = 0.9")
plt.legend()
plt.show()

#### Разбор кода

Одинаковый seed делает сравнение особенно прозрачным: меняется масштаб одной и той же базовой псевдослучайной последовательности. Стандартное отклонение и ширина распределения реагируют на изменение `sigma` непосредственно.

**Итог раздела.** Собственные функции позволяют отделить повторяющийся анализ данных, а параметрический эксперимент связывает изменение входного параметра с наблюдаемыми статистическими последствиями.

# 7. Самостоятельная работа

Для всех заданий используйте строку с номером своего варианта из файла `lab03_functions_noise_variants_v2.py`. Номер строки равен номеру варианта. Полные решения находятся только в базе задач.

## С1. Параметризованная функция

По коэффициентам варианта напишите функцию $P(x)=ax^2+bx+c$. Проверьте её в точке `x_check` и ещё в одной точке по выбору. Одно значение сначала вычислите вручную.

## С2. Дискретное представление функции

Используйте функцию из С1. Создайте `np.linspace` от `x_left` до `x_right` из `N` точек, вычислите $P(x)$, постройте график и выведите фактический шаг сетки.

## С3. Стандартное отклонение как собственная функция

Сформируйте массив `data = data_center + data_scale * [-2,-1,-1,0,1,2,3]`. Напишите `std_manual(data)`, не используя `np.std`, затем сравните результат с `np.std`.

## С4. Воспроизводимая нормальная случайная выборка

Создайте два генератора с `seed` своего варианта и две нормальные выборки размера `M` с $\mu=0$ и `sigma`. Проверьте, что выборки совпадают, и вычислите `mean` и `std` одной выборки.

## С5. Зашумлённые данные и остаток

Табулируйте $P(x)$ на сетке варианта, добавьте нормальный шум с `sigma` и `seed`, вычислите остаток $r=y_{noisy}-y_{clean}$ и сравните `std(r)` с заданным `sigma`.

## С6. Квантили и межквартильный размах

Для остатка из С5 вычислите $Q_{0.25}$, медиану, $Q_{0.75}$ и IQR. Сравните медиану со средним остатка.

## С7. Гистограмма распределения

Постройте гистограмму остатка из С5 с `bins` своего варианта и затем с `2*bins`. Кратко опишите, что изменилось в представлении, а что осталось неизменным.

## С8. Функция статистического описания

Напишите `describe_data(data)`, которая возвращает `mean`, `std`, `min`, `max`, $Q_{0.25}$, медиану и $Q_{0.75}$. Примените её к остатку из С5.

## И1. Анализ зашумлённых отсчётов функции

По паспорту варианта задайте $P(x)$, получите дискретное представление, добавьте шум, постройте чистые и зашумлённые данные, вычислите статистики остатка функцией `describe_data` и его гистограмму. Сформулируйте 2–3 числовых вывода.

## И2. Сравнение уровней шума

Для той же $P(x)$ создайте два набора наблюдений с `sigma` и `2*sigma` при одинаковом `seed`. Сравните статистики остатков и гистограммы; объясните, какие характеристики должны измениться примерно вдвое.

# 8. Контрольные задания

Конкретный набор заданий выбирает преподаватель. Для каждого задания используйте строку своего варианта из `lab03_functions_noise_variants_v2.py`.

## K1. Чтение функции

Не запуская код, вычислите значение P(x_check) для функции с коэффициентами варианта, затем проверьте программой.

## K2. Исправление `return`

В готовой функции P(x) вычисляется правильная переменная y, но возвращается x. Исправьте функцию и проверьте в x_check.

## K3. Функция с параметрами

Напишите функцию P(x,a,b,c) по коэффициентам варианта и вычислите P(x_check).

## K4. Массив как аргумент функции

Передайте в P NumPy-массив из трёх значений x_check-1, x_check, x_check+1 и получите массив результатов без явного цикла.

## K5. `arange` или `linspace`

Для условия «от x_left до x_right включительно получить N равномерных точек» выберите подходящую функцию и создайте сетку.

## K6. Шаг `linspace`

Сначала вручную вычислите шаг сетки от x_left до x_right из N точек, затем проверьте разностью соседних элементов.

## K7. Стандартное отклонение по определению

Для массива data_center + data_scale*[-2,-1,-1,0,1,2,3] вычислите std без np.std, затем проверьте.

## K8. Воспроизводимый шум

Создайте два генератора с одинаковым seed и убедитесь, что первые нормальные выборки размера M совпадают.

## K9. Отладка размера шума

В коде шум создаётся длины N-1 и не складывается с массивом P(x) длины N. Исправьте размер шума.

## K10. Квантили и IQR

Для нормальной выборки варианта вычислите Q0.25, медиану, Q0.75 и IQR.

## K11. Гистограмма распределения

Сгенерируйте нормальную выборку варианта и постройте гистограмму с bins интервалами. Объясните, что показывает высота столбца.

## K12. Интегральный анализ остатка

Получите P(x) на linspace, добавьте шум, вычислите остаток, его mean/std/квартили и постройте гистограмму.

# Заключение

В этой работе программная функция стала основным способом организовать повторяющееся вычисление, а статистические характеристики были связаны с их математическими определениями до применения готовых библиотечных функций.

Основная схема работы:

$$\text{функция}\rightarrow\text{равномерная сетка}\rightarrow\text{дискретное представление}\rightarrow\text{разброс}\rightarrow\text{случайная выборка}\rightarrow\text{зашумлённые данные}\rightarrow\text{остаток}\rightarrow\text{числовое и графическое описание}.$$

После работы студент должен уметь:

- оформлять формулу и повторяющийся алгоритм как функцию Python;
- передавать NumPy-массив в собственную функцию;
- выбирать между `np.arange` и `np.linspace` по смыслу постановки;
- вычислять среднее, дисперсию и стандартное отклонение по определению;
- создавать воспроизводимые случайные выборки и нормальный шум;
- анализировать остатки через статистики, квантили и гистограмму;
- собирать несколько характеристик в повторно используемую функцию `describe_data`.

Следующая работа может вводить собственно дискретизацию и квантование: здесь сетка пока рассматривается как способ получить конечный набор аргументов, без введения физической частоты дискретизации.